# Fase 3 — Análisis estadístico

Lee `results/metrics.csv` y ejecuta:

| Prueba | Datos | Comparación |
|--------|-------|-------------|
| Wilcoxon pareado | `same_seed` | original vs modificado, por semilla |
| Mann-Whitney U | `diff_seed` | original vs modificado, independiente |

Para cada N ∈ {50, 100, 150, 200} y cada métrica (HV, GD, IGD).  
Corrección de Holm-Bonferroni sobre los p-valores de cada métrica.  
Tamaño del efecto: Vargha-Delaney Â₁₂ (via `pingouin`).

In [1]:
import pandas as pd
import numpy as np
import pingouin as pg
from statsmodels.stats.multitest import multipletests

RESULTS_BASE = '../results'
N_VALUES     = [50, 100, 150, 200]
METRICS      = ['HV', 'GD', 'IGD']

In [2]:
df = pd.read_csv(f'{RESULTS_BASE}/metrics.csv')
print(f'Corridas cargadas: {len(df)}')
df.groupby(['variante', 'seed_mode', 'N']).size().rename('n').to_frame()

Corridas cargadas: 480


n
variante seed_mode N      
modified diff_seed 50   30
                   100  30
                   150  30
                   200  30
         same_seed 50   30
                   100  30
                   150  30
                   200  30
original diff_seed 50   30
                   100  30
                   150  30
                   200  30
         same_seed 50   30
                   100  30
                   150  30
                   200  30

## 1. Pruebas estadísticas por métrica

In [3]:
def run_tests(df, metric):
    rows = []
    for n in N_VALUES:
        orig_same = df[(df.variante == 'original')  & (df.seed_mode == 'same_seed') & (df.N == n)]
        mod_same  = df[(df.variante == 'modified')  & (df.seed_mode == 'same_seed') & (df.N == n)]
        orig_diff = df[(df.variante == 'original')  & (df.seed_mode == 'diff_seed') & (df.N == n)]
        mod_diff  = df[(df.variante == 'modified')  & (df.seed_mode == 'diff_seed') & (df.N == n)]

        # ── Wilcoxon pareado (same_seed) ──────────────────────────────────
        if len(orig_same) > 0 and len(mod_same) > 0:
            merged = orig_same.merge(mod_same, on='seed', suffixes=('_orig', '_mod'))
            if len(merged) >= 2:
                try:
                    res_w = pg.wilcoxon(merged[f'{metric}_orig'], merged[f'{metric}_mod'])
                    p_w   = float(res_w['p_val'].iloc[0])
                    a12_w = float(res_w['CLES'].iloc[0]) if 'CLES' in res_w.columns else np.nan
                except ValueError:
                    # Todas las diferencias pareadas son cero (orig == mod en todos los seeds)
                    p_w, a12_w = 1.0, 0.5
            else:
                p_w, a12_w = np.nan, np.nan
        else:
            p_w, a12_w = np.nan, np.nan

        rows.append({'N': n, 'test': 'Wilcoxon', 'seed_mode': 'same_seed',
                     'p_valor': p_w, 'A12': a12_w})

        # ── Mann-Whitney U (diff_seed) ────────────────────────────────────
        if len(orig_diff) >= 2 and len(mod_diff) >= 2:
            res_mw = pg.mwu(orig_diff[metric], mod_diff[metric], alternative='two-sided')
            p_mw   = float(res_mw['p_val'].iloc[0])
            a12_mw = float(res_mw['CLES'].iloc[0]) if 'CLES' in res_mw.columns else np.nan
        else:
            p_mw, a12_mw = np.nan, np.nan

        rows.append({'N': n, 'test': 'Mann-Whitney', 'seed_mode': 'diff_seed',
                     'p_valor': p_mw, 'A12': a12_mw})

    result_df = pd.DataFrame(rows)

    # ── Holm-Bonferroni sobre todos los p-valores de esta métrica ─────────
    valid_mask = result_df['p_valor'].notna()
    if valid_mask.sum() > 0:
        pvals = result_df.loc[valid_mask, 'p_valor'].values
        reject, pvals_corr, _, _ = multipletests(pvals, method='holm')
        result_df.loc[valid_mask, 'p_corr_holm'] = pvals_corr
        result_df.loc[valid_mask, 'reject_H0']   = reject
    else:
        result_df['p_corr_holm'] = np.nan
        result_df['reject_H0']   = np.nan

    result_df.insert(0, 'metrica', metric)
    return result_df

all_results = pd.concat([run_tests(df, m) for m in METRICS], ignore_index=True)
all_results

,metrica,N,test,seed_mode,p_valor,A12,p_corr_holm,reject_H0
0,HV,50,Wilcoxon,same_seed,1.862645e-09,0.000000,7.450581e-09,True
1,HV,50,Mann-Whitney,diff_seed,3.019859e-11,0.000000,2.414373e-10,True
2,HV,100,Wilcoxon,same_seed,1.862645e-09,0.000000,7.450581e-09,True
3,HV,100,Mann-Whitney,diff_seed,3.017967e-11,0.000000,2.414373e-10,True
4,HV,150,Wilcoxon,same_seed,1.862645e-09,0.000000,7.450581e-09,True
5,HV,150,Mann-Whitney,diff_seed,3.019859e-11,0.000000,2.414373e-10,True
6,HV,200,Wilcoxon,same_seed,1.862645e-09,0.000000,7.450581e-09,True
7,HV,200,Mann-Whitney,diff_seed,3.019859e-11,0.000000,2.414373e-10,True
8,GD,50,Wilcoxon,same_seed,1.852948e-02,0.360000,5.558844e-02,False
9,GD,50,Mann-Whitney,diff_seed,5.011437e-01,0.448889,5.011437e-01,False


## 2. Guardar resultados

In [4]:
out_path = f'{RESULTS_BASE}/statistical_results.csv'
all_results.to_csv(out_path, index=False)
print(f'Guardado: {out_path}')

Guardado: ../results/statistical_results.csv


## 3. Tabla de resultados por métrica

In [5]:
def fmt_pval(p):
    if pd.isna(p): return 'N/A'
    if p < 0.001: return '< 0.001'
    return f'{p:.4f}'

for metric in METRICS:
    sub = all_results[all_results.metrica == metric].copy()
    sub['p_valor_fmt']    = sub['p_valor'].map(fmt_pval)
    sub['p_corr_holm_fmt'] = sub['p_corr_holm'].map(fmt_pval)
    sub['A12_fmt']        = sub['A12'].map(lambda x: f'{x:.4f}' if pd.notna(x) else 'N/A')
    sub['significativo']  = sub['reject_H0'].map(lambda x: '✓' if x == True else ('✗' if x == False else 'N/A'))
    print(f'\n{'='*60}')
    print(f'Métrica: {metric}')
    print('='*60)
    display_cols = ['N', 'test', 'seed_mode', 'p_valor_fmt', 'p_corr_holm_fmt', 'A12_fmt', 'significativo']
    print(sub[display_cols].to_string(index=False))


Métrica: HV
  N         test seed_mode p_valor_fmt p_corr_holm_fmt A12_fmt significativo
 50     Wilcoxon same_seed     < 0.001         < 0.001  0.0000             ✓
 50 Mann-Whitney diff_seed     < 0.001         < 0.001  0.0000             ✓
100     Wilcoxon same_seed     < 0.001         < 0.001  0.0000             ✓
100 Mann-Whitney diff_seed     < 0.001         < 0.001  0.0000             ✓
150     Wilcoxon same_seed     < 0.001         < 0.001  0.0000             ✓
150 Mann-Whitney diff_seed     < 0.001         < 0.001  0.0000             ✓
200     Wilcoxon same_seed     < 0.001         < 0.001  0.0000             ✓
200 Mann-Whitney diff_seed     < 0.001         < 0.001  0.0000             ✓

Métrica: GD
  N         test seed_mode p_valor_fmt p_corr_holm_fmt A12_fmt significativo
 50     Wilcoxon same_seed      0.0185          0.0556  0.3600             ✗
 50 Mann-Whitney diff_seed      0.5011          0.5011  0.4489             ✗
100     Wilcoxon same_seed      0.0087          0.

## 4. Interpretación de Â₁₂

| Â₁₂ | Interpretación |
|------|---------------|
| ≈ 0.5 | Sin efecto (equivalente) |
| 0.56 – 0.64 | Efecto pequeño |
| 0.64 – 0.71 | Efecto mediano |
| > 0.71 | Efecto grande |

Â₁₂ > 0.5 indica que el **modificado** supera al original con esa probabilidad (si comparamos original primero).

In [6]:
pivot = all_results.pivot_table(
    index=['metrica', 'N'],
    columns='test',
    values=['p_valor', 'p_corr_holm', 'A12', 'reject_H0']
)
pivot

A12             p_corr_holm                     p_valor  \
test        Mann-Whitney  Wilcoxon  Mann-Whitney      Wilcoxon  Mann-Whitney   
metrica N                                                                      
GD      50      0.448889  0.360000  5.011437e-01  5.558844e-02  5.011437e-01   
        100     0.268889  0.284444  1.078165e-02  3.482185e-02  2.156330e-03   
        150     0.184444  0.278889  1.940800e-04  5.558844e-02  2.772572e-05   
        200     0.212222  0.148889  7.949706e-04  9.746850e-05  1.324951e-04   
HV      50      0.000000  0.000000  2.414373e-10  7.450581e-09  3.019859e-11   
        100     0.000000  0.000000  2.414373e-10  7.450581e-09  3.017967e-11   
        150     0.000000  0.000000  2.414373e-10  7.450581e-09  3.019859e-11   
        200     0.000000  0.000000  2.414373e-10  7.450581e-09  3.019859e-11   
IGD     50      0.561111  0.528889  1.000000e+00  1.000000e+00  4.203863e-01   
        100     0.485556  0.371111  1.000000e+00  2.929763e-01  8.533797e-01   
        150     0.314444  0.374444  9.682134e-02  2.266921e-01  1.383162e-02   
        200     0.343333  0.271111  2.266921e-01  1.266627e-02  3.778202e-02   

                             reject_H0           
test             Wilcoxon Mann-Whitney Wilcoxon  
metrica N                                        
GD      50   1.852948e-02          0.0      0.0  
        100  8.705463e-03          1.0      1.0  
        150  1.966082e-02          1.0      0.0  
        200  1.218356e-05          1.0      1.0  
HV      50   1.862645e-09          1.0      1.0  
        100  1.862645e-09          1.0      1.0  
        150  1.862645e-09          1.0      1.0  
        200  1.862645e-09          1.0      1.0  
IGD     50   8.712078e-01          0.0      0.0  
        100  7.324407e-02          0.0      0.0  
        150  3.841842e-02          0.0      0.0  
        200  1.583284e-03          0.0      1.0